# MLP for CO₂ Adsorption: Baselines & HPO

**Splits:** topology · metal · random.  
**Target:** `co2_mol_kg_0.1bar` (raw mol/kg).  
**HPO** is handled by `run_hpo_geo_rac.py` (parallelised, skips completed experiments).  
Results are loaded here from `results/mlp/mlp_summary.csv`.

In [2]:
import json, sys, time, warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from src.data_utils import load_experiment

In [3]:
MODELS_DIR  = ROOT / 'models' / 'mlp'
RESULTS_DIR = ROOT / 'results' / 'mlp'

TARGET_IDX   = 2 # co2_mol_kg_0.1bar
SPLITS       = ['topology', 'metal', 'random'] # smallest first
RANDOM_STATE = 42
CV_FOLDS     = 5
N_TRIALS     = 30 # Optuna trials per HPO run

FIXED_MLP = dict(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu', solver='adam',
    alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
    max_iter=500, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

HIDDEN_CONFIGS = [
    (64,), (128,), (256,), (512,),
    (128, 64), (256, 128), (512, 256),
    (256, 128, 64), (512, 256, 128),
]

GEO_SETS      = [('geo_all', 'Geo (all)', 6),       ('geo_decorr', 'Geo (decorr)', 5)]
RAC_SETS      = [('rac_all', 'RAC (all)', 81),       ('rac_decorr', 'RAC (decorr)', 45)]
COMBINED_SETS = [('combined_all', 'Combined (all)', 87), ('combined_decorr', 'Combined (decorr)', 50)]

In [4]:
def get_arrays(split, feature_set, encoding='label'):
    #Load one (split x feature_set) experiment. Returns 1-D y (TARGET_IDX only)
    X_tr, X_te, y_tr, y_te = load_experiment(split, feature_set, encoding=encoding)
    return X_tr, y_tr[:, TARGET_IDX], X_te, y_te[:, TARGET_IDX]


def make_pipe(mlp_kw=None):
    #Pipeline: SimpleImputer -> StandardScaler -> MLPRegressor
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('mlp', MLPRegressor(**(mlp_kw or FIXED_MLP))),
    ])


def calc_metrics(y_true, y_pred):
    return dict(
        RMSE = float(np.sqrt(mean_squared_error(y_true, y_pred))),
        MAE  = float(mean_absolute_error(y_true, y_pred)),
        R2   = float(r2_score(y_true, y_pred)),
    )


def report(feature_set, split, n_feat, n_iter, m):
    print(f'[{feature_set:<20s}] {split:<9s}  feat={n_feat:>3d}  iter={n_iter:>4d}', flush=True)
    print(f'  test — RMSE {m["RMSE"]:.4f}  MAE {m["MAE"]:.4f}  R2 {m["R2"]:.4f}', flush=True)


def save_model(pipe, tag):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, MODELS_DIR / f'{tag}.joblib')


def train_eval(split, feature_set, n_feat, phase_label, mlp_kw=None, encoding='label'):
    #Fit pipeline on full train set, evaluate on test, save model
    X_tr, y_tr, X_te, y_te = get_arrays(split, feature_set, encoding=encoding)
    pipe = make_pipe(mlp_kw)
    pipe.fit(X_tr, y_tr)
    m     = calc_metrics(y_te, pipe.predict(X_te))
    n_iter = pipe.named_steps['mlp'].n_iter_
    report(phase_label, split, n_feat, n_iter, m)
    tag = f"{feature_set}__{split}__fixed"
    save_model(pipe, tag)
    return dict(
        feature_set=feature_set, split=split, method='fixed',
        n_features=n_feat, best_cv_RMSE=float('nan'), n_iter=n_iter,
        **{f'test_{k}': v for k, v in m.items()},
    )

### Mean predictor (floor)

Predicts the training mean. Sets the absolute floor.

In [4]:
results = []

for split in SPLITS:
    X_tr, y_tr, X_te, y_te = get_arrays(split, 'geo_all')
    dummy = DummyRegressor(strategy='mean')
    dummy.fit(X_tr, y_tr)
    m = calc_metrics(y_te, dummy.predict(X_te))
    report('Mean pred', split, 0, 0, m)
    print(flush=True)
    results.append(dict(
        feature_set='none', split=split, method='mean_pred',
        n_features=0, best_cv_RMSE=float('nan'), n_iter=0,
        **{f'test_{k}': v for k, v in m.items()},
    ))

[Mean pred           ] topology   feat=  0  iter=   0
  test — RMSE 0.6615  MAE 0.5432  R2 -0.0419

[Mean pred           ] metal      feat=  0  iter=   0
  test — RMSE 0.6738  MAE 0.4909  R2 -0.0077

[Mean pred           ] random     feat=  0  iter=   0
  test — RMSE 0.6693  MAE 0.4740  R2 -0.0001



### Categorical baseline

`topology` + `metal_node`, target-encoded (2 features).

In [5]:
for split in SPLITS:
    results.append(train_eval(split, 'baseline', 2, 'Baseline', encoding='target'))
    print(flush=True)

[Baseline            ] topology   feat=  2  iter=  35
  test — RMSE 0.6434  MAE 0.5245  R2 0.0141

[Baseline            ] metal      feat=  2  iter=  55
  test — RMSE 0.6667  MAE 0.4684  R2 0.0135

[Baseline            ] random     feat=  2  iter=  30
  test — RMSE 0.6593  MAE 0.4607  R2 0.0296



### Phase 2 Geometric features

All (6) vs decorrelated (5).

In [6]:
for feat_set, label, n_feat in GEO_SETS:
    for split in SPLITS:
        results.append(train_eval(split, feat_set, n_feat, label))
    print(flush=True)

[Geo (all)           ] topology   feat=  6  iter=  75
  test — RMSE 0.3749  MAE 0.2170  R2 0.6653
[Geo (all)           ] metal      feat=  6  iter=  82
  test — RMSE 0.3677  MAE 0.2343  R2 0.6999
[Geo (all)           ] random     feat=  6  iter=  98
  test — RMSE 0.3724  MAE 0.2286  R2 0.6904

[Geo (decorr)        ] topology   feat=  5  iter=  73
  test — RMSE 0.3174  MAE 0.1828  R2 0.7601
[Geo (decorr)        ] metal      feat=  5  iter=  77
  test — RMSE 0.3772  MAE 0.2438  R2 0.6842
[Geo (decorr)        ] random     feat=  5  iter=  82
  test — RMSE 0.3871  MAE 0.2326  R2 0.6654



### Phase 3: RAC features

All (81) vs decorrelated (45).

In [7]:
for feat_set, label, n_feat in RAC_SETS:
    for split in SPLITS:
        results.append(train_eval(split, feat_set, n_feat, label))
    print(flush=True)

[RAC (all)           ] topology   feat= 81  iter=  84
  test — RMSE 0.6232  MAE 0.3737  R2 0.0752
[RAC (all)           ] metal      feat= 81  iter= 123
  test — RMSE 0.6640  MAE 0.4824  R2 0.0214
[RAC (all)           ] random     feat= 81  iter= 134
  test — RMSE 0.6153  MAE 0.4153  R2 0.1547

[RAC (decorr)        ] topology   feat= 45  iter= 119
  test — RMSE 0.6652  MAE 0.3935  R2 -0.0536
[RAC (decorr)        ] metal      feat= 45  iter=  54
  test — RMSE 0.6346  MAE 0.4222  R2 0.1061
[RAC (decorr)        ] random     feat= 45  iter=  79
  test — RMSE 0.6173  MAE 0.4187  R2 0.1490



### Phase 4: Combined features, fixed architecture `(256, 128, 64)`

All (87) vs decorrelated (50).

In [8]:
for feat_set, label, n_feat in COMBINED_SETS:
    for split in SPLITS:
        results.append(train_eval(split, feat_set, n_feat, label))
    print(flush=True)

[Combined (all)      ] topology   feat= 87  iter=  82
  test — RMSE 0.3497  MAE 0.1968  R2 0.7088
[Combined (all)      ] metal      feat= 87  iter= 103
  test — RMSE 0.3579  MAE 0.2141  R2 0.7156
[Combined (all)      ] random     feat= 87  iter=  76
  test — RMSE 0.3189  MAE 0.1872  R2 0.7729

[Combined (decorr)   ] topology   feat= 50  iter=  98
  test — RMSE 0.3333  MAE 0.1982  R2 0.7355
[Combined (decorr)   ] metal      feat= 50  iter=  95
  test — RMSE 0.3782  MAE 0.2321  R2 0.6825
[Combined (decorr)   ] random     feat= 50  iter=  88
  test — RMSE 0.3255  MAE 0.1926  R2 0.7634



### HPO

HPO for all feature sets is in `run_hpo_geo_rac.py`
Results are appended to `results/mlp/mlp_summary.csv` and loaded in the Summary section below.

### CV RMSE backfill:

Computes 5-fold CV RMSE for all fixed-arch experiments using the same fixed architecture. 

In [ ]:
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for r in results:
    if not np.isnan(r['best_cv_RMSE']):
        continue  # already filled (HPO rows)

    fs  = r['feature_set']
    sp  = r['split']
    enc = 'target' if fs == 'baseline' else 'label'

    ref_fs = 'geo_all' if fs == 'none' else fs
    X_tr, y_tr, _, _ = get_arrays(sp, ref_fs, encoding=enc)

    if fs == 'none':
        estimator = DummyRegressor(strategy='mean')
    else:
        estimator = make_pipe()

    scores = cross_val_score(estimator, X_tr, y_tr, cv=kf, scoring='neg_mean_squared_error')
    cv_rmse = float(np.sqrt(-scores.mean()))
    r['best_cv_RMSE'] = cv_rmse
    print(f'{fs:<20s}  {sp:<9s}  cv_rmse={cv_rmse:.4f}', flush=True)

none                  topology   cv_rmse=0.6708
none                  metal      cv_rmse=0.6699
none                  random     cv_rmse=0.6709
baseline              topology   cv_rmse=0.6614
baseline              metal      cv_rmse=0.6603
baseline              random     cv_rmse=0.6618
geo_all               topology   cv_rmse=0.3800
geo_all               metal      cv_rmse=0.3928
geo_all               random     cv_rmse=0.3829
geo_decorr            topology   cv_rmse=0.3904
geo_decorr            metal      cv_rmse=0.4022
geo_decorr            random     cv_rmse=0.3919


### Save & Summary

In [13]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RESULTS_DIR / 'mlp_summary.csv'
df = pd.DataFrame(results)
df.to_csv(SUMMARY_PATH, index=False)
print(f'Saved {len(df)} rows -> {SUMMARY_PATH}', flush=True)

Saved 21 rows -> /Users/hannah/Desktop/ethz/Semester6/dc-mof-project/results/mlp/mlp_summary.csv


In [14]:
df = pd.read_csv(RESULTS_DIR / 'mlp_summary.csv')

NO_SUFFIX = {'none', 'baseline'}
df['label'] = df.apply(
    lambda r: r['feature_set'] if r['feature_set'] in NO_SUFFIX
              else f"{r['feature_set']} [{r['method']}]",
    axis=1,
)

LABEL_ORDER = [
    'none', 'baseline',
    'geo_decorr [fixed]', 'geo_decorr [optuna]',
    'geo_all [fixed]', 'geo_all [optuna]',
    'rac_decorr [fixed]', 'rac_decorr [optuna]',
    'rac_all [fixed]', 'rac_all [optuna]',
    'combined_decorr [fixed]', 'combined_decorr [optuna]',
    'combined_all [fixed]', 'combined_all [optuna]',
]

METRICS = [
    ('best_cv_RMSE', 'CV RMSE (5-fold)'),
    ('test_RMSE', 'Test RMSE'),
    ('test_MAE', 'Test MAE'),
    ('test_R2', 'Test R2'),
]

for col, title in METRICS:
    p = df.pivot_table(index='label', columns='split', values=col, aggfunc='first')
    p = p.reindex([l for l in LABEL_ORDER if l in p.index])
    print(title)
    display(p.round(4))
    print()

CV RMSE (5-fold)


split,metal,random,topology
label,,,
baseline,0.6603,0.6618,0.6614
geo_decorr [optuna],0.4013,0.3927,0.3911
geo_all [optuna],0.3923,0.3819,0.3807
rac_decorr [optuna],0.6231,0.6201,0.6197
rac_all [optuna],0.6234,0.6208,0.6194
combined_decorr [optuna],0.3360,0.3258,0.3252
combined_all [optuna],0.3300,0.3203,0.3187



Test RMSE


split,metal,random,topology
label,,,
baseline,0.6667,0.6593,0.6434
geo_decorr [optuna],0.3746,0.3866,0.3230
geo_all [optuna],0.3709,0.3731,0.3362
rac_decorr [optuna],0.6537,0.6159,0.6445
rac_all [optuna],0.6384,0.6157,0.6296
combined_decorr [optuna],0.3697,0.3216,0.3463
combined_all [optuna],0.3680,0.3140,0.3454



Test MAE


split,metal,random,topology
label,,,
baseline,0.4684,0.4607,0.5245
geo_decorr [optuna],0.2404,0.2335,0.1840
geo_all [optuna],0.2398,0.2235,0.2017
rac_decorr [optuna],0.4465,0.4183,0.3811
rac_all [optuna],0.4190,0.4153,0.3538
combined_decorr [optuna],0.2182,0.1877,0.2194
combined_all [optuna],0.2186,0.1863,0.1897



Test R2


split,metal,random,topology
label,,,
baseline,0.0135,0.0296,0.0141
geo_decorr [optuna],0.6885,0.6662,0.7516
geo_all [optuna],0.6947,0.6891,0.7308
rac_decorr [optuna],0.0513,0.1531,0.0109
rac_all [optuna],0.0955,0.1534,0.0562
combined_decorr [optuna],0.6967,0.7690,0.7145
combined_all [optuna],0.6995,0.7798,0.7159


### Restore existing results

In [5]:
results = pd.read_csv(RESULTS_DIR / 'mlp_summary.csv').to_dict('records')
studies = {}
print(f'Restored {len(results)} results from {RESULTS_DIR / "mlp_summary.csv"}', flush=True)
for r in results:
    print(f"  {r['feature_set']:<20s}  {r['split']:<9s}  {r['method']}", flush=True)

Restored 21 results from /Users/hannah/Desktop/ethz/Semester6/dc-mof-project/results/mlp/mlp_summary.csv
  baseline              metal      fixed
  baseline              random     fixed
  geo_decorr            topology   optuna
  geo_decorr            metal      optuna
  geo_decorr            random     optuna
  geo_all               topology   optuna
  geo_all               metal      optuna
  geo_all               random     optuna
  rac_decorr            topology   optuna
  rac_decorr            metal      optuna
  rac_decorr            random     optuna
  rac_all               topology   optuna
  rac_all               metal      optuna
  rac_all               random     optuna
  combined_decorr       topology   optuna
  combined_decorr       metal      optuna
  combined_decorr       random     optuna
  combined_all          topology   optuna
  combined_all          metal      optuna
  combined_all          random     optuna
  baseline              topology   fixed


## Results overview: mlp_results_final.csv

In [6]:
df_res = pd.read_csv(ROOT / 'results' / 'mlp' / 'mlp_results_final.csv')

FEATURE_SETS = ['baseline', 'geo_decorr', 'geo_all', 'rac_decorr', 'rac_all',
                'combined_decorr', 'combined_all']

# CV RMSE
print("CV RMSE (5-fold on train set)")
pivot_cv = df_res.pivot_table(index="feature_set", columns="split", values="best_cv_RMSE")
pivot_cv = pivot_cv.reindex(FEATURE_SETS)
display(pivot_cv.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test RMSE
print("\nTest RMSE")
pivot_test_rmse = df_res.pivot_table(index="feature_set", columns="split", values="test_RMSE")
pivot_test_rmse = pivot_test_rmse.reindex(FEATURE_SETS)
display(pivot_test_rmse.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test MAE
print("\nTest MAE")
pivot_test_mae = df_res.pivot_table(index="feature_set", columns="split", values="test_MAE")
pivot_test_mae = pivot_test_mae.reindex(FEATURE_SETS)
display(pivot_test_mae.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test R^2
print("\nTest R^2")
pivot_r2 = df_res.pivot_table(index="feature_set", columns="split", values="test_R2")
pivot_r2 = pivot_r2.reindex(FEATURE_SETS)
display(pivot_r2.style.format("{:.4f}").background_gradient(cmap="RdYlGn"))

CV RMSE (5-fold on train set)


split,metal,random,topology
feature_set,,,
baseline,0.6603,0.6618,0.6614
geo_decorr,0.4013,0.3927,0.3911
geo_all,0.3923,0.3819,0.3807
rac_decorr,0.6231,0.6201,0.6197
rac_all,0.6234,0.6208,0.6194
combined_decorr,0.3360,0.3258,0.3252
combined_all,0.3300,0.3203,0.3187



Test RMSE


split,metal,random,topology
feature_set,,,
baseline,0.6667,0.6593,0.6434
geo_decorr,0.3746,0.3866,0.3230
geo_all,0.3709,0.3731,0.3362
rac_decorr,0.6537,0.6159,0.6445
rac_all,0.6384,0.6157,0.6296
combined_decorr,0.3697,0.3216,0.3463
combined_all,0.3680,0.3140,0.3454



Test MAE


split,metal,random,topology
feature_set,,,
baseline,0.4684,0.4607,0.5245
geo_decorr,0.2404,0.2335,0.1840
geo_all,0.2398,0.2235,0.2017
rac_decorr,0.4465,0.4183,0.3811
rac_all,0.4190,0.4153,0.3538
combined_decorr,0.2182,0.1877,0.2194
combined_all,0.2186,0.1863,0.1897



Test R^2


split,metal,random,topology
feature_set,,,
baseline,0.0135,0.0296,0.0141
geo_decorr,0.6885,0.6662,0.7516
geo_all,0.6947,0.6891,0.7308
rac_decorr,0.0513,0.1531,0.0109
rac_all,0.0955,0.1534,0.0562
combined_decorr,0.6967,0.7690,0.7145
combined_all,0.6995,0.7798,0.7159
